## Feature Engineering

將原始臨床資料轉換為模型可用的特徵，處理連續變數與類別變數的缺失值問題。

### 策略
將每位病人的資料切割為 10 小時滑動視窗，預測下一小時是否發生敗血症。

| 特徵類型 | 欄位 | 處理方式 |
|---------|------|---------|
| 連續時序特徵（缺失 < 15%） | HR, MAP, O2Sat, SBP, Resp | 保留完整 10 小時時序資料 |
| 其他變數（缺失 > 15%） | Temp, Glucose, Lactate 等 | 取 10 小時視窗的中位數 |

In [ ]:
import pandas as pd
import numpy as np
import pdb
import os
import shutil
import gdown

In [ ]:
# 建立 feats 資料夾，若已存在則先刪除重建
try:
    if os.path.exists('feats'):
        shutil.rmtree('feats')
    os.makedirs('feats')
except Exception as e:
    print(e)

In [ ]:
# 從作者的 Google Drive 下載已處理好的 combined.pkl
file_id = '1AmIJQ2oo7Cy1w32T8d1v-rXiJKM0wZE-'
gdown.download(f'https://drive.google.com/uc?id={file_id}', './combined.pkl', quiet=False)

# 讀取資料
df = pd.read_pickle('combined.pkl')

Downloading...
From (original): https://drive.google.com/uc?id=1AmIJQ2oo7Cy1w32T8d1v-rXiJKM0wZE-
From (redirected): https://drive.google.com/uc?id=1AmIJQ2oo7Cy1w32T8d1v-rXiJKM0wZE-&confirm=t&uuid=3e0f5e2b-3e0e-46ce-aaf5-c7528f094efd
To: /content/combined.pkl
100%|██████████| 517M/517M [00:05<00:00, 97.1MB/s]


In [ ]:
# 計算每個欄位的缺失值比例
print('Percentage Missing:')
print(df.isna().sum()/len(df))

Percentage Missing:
HR                  0.098826
O2Sat               0.130611
Temp                0.661627
SBP                 0.145770
MAP                 0.124513
DBP                 0.313459
Resp                0.153546
EtCO2               0.962868
BaseExcess          0.945790
HCO3                0.958106
FiO2                0.916658
pH                  0.930697
PaCO2               0.944401
SaO2                0.965494
AST                 0.983776
BUN                 0.931344
Alkalinephos        0.983932
Calcium             0.941161
Chloride            0.954603
Creatinine          0.939044
Bilirubin_direct    0.998074
Glucose             0.828943
Lactate             0.973299
Magnesium           0.936896
Phosphate           0.959863
Potassium           0.906891
Bilirubin_total     0.985092
TroponinI           0.990477
Hct                 0.911460
Hgb                 0.926176
PTT                 0.970559
WBC                 0.935932
Fibrinogen          0.993402
Platelets           0.9

In [ ]:
# 刪除不需要的欄位
# Unit2 與 Unit1 互斥，ICULOS 只是時間索引
cols_to_drop = ['Unit2', 'ICULOS']
df = df.drop(cols_to_drop, axis=1)

# 缺失值 < 15% 的連續欄位，保留為時序特徵
cols_cont = ['HR', 'MAP', 'O2Sat', 'SBP', 'Resp']

# 缺失值 > 15% 的連續欄位，另外處理
cols_to_bin = ['Unit1', 'Gender', 'HospAdmTime', 'Age', 'DBP', 'Temp',
               'Glucose', 'Potassium', 'Hct', 'FiO2', 'Hgb', 'pH', 'BUN',
               'WBC', 'Magnesium', 'Creatinine', 'Platelets', 'Calcium', 'PaCO2',
               'BaseExcess', 'Chloride', 'HCO3', 'Phosphate', 'EtCO2', 'SaO2',
               'PTT', 'Lactate', 'AST', 'Alkalinephos', 'Bilirubin_total',
               'TroponinI', 'Fibrinogen', 'Bilirubin_direct']

In [ ]:
# 取得所有不重複的病人 ID 並隨機打亂順序
patients_training_data = df['patient'].unique()
np.random.shuffle(patients_training_data)

# 保留除最後 6000 個病人以外的資料作為訓練集
patients_training_data = patients_training_data[0:-6000]

# 計算訓練集的 mean 和 std，供後續標準化使用
df_mean_std = df[df['patient'].isin(patients_training_data)].describe().loc[['mean', 'std']]
df_mean_std.to_pickle('mean_std_scaling.pkl')

In [ ]:
# 計算訓練集中敗血症（SepsisLabel=1）的筆數
print('Number of positive training examples:')
sum(df[df['patient'].isin(patients_training_data)]['SepsisLabel']==1)

Number of positive training examples:


23768

In [ ]:
# 初始化計數器和暫存清單
save_count = 0
windowed_df_list = []

# 依病人 ID 分組
grouped_by_patient = df.groupby('patient')

# 逐一處理每個病人的資料
for patient, group in grouped_by_patient:
    group = group.reset_index(drop=True)

    # 補缺失值：先用後面的值填（bfill），再用前面的值填（ffill）
    group = group.assign(HR=group['HR'].bfill().ffill())
    group = group.assign(MAP=group['MAP'].bfill().ffill())
    group = group.assign(O2Sat=group['O2Sat'].bfill().ffill())
    group = group.assign(SBP=group['SBP'].bfill().ffill())
    group = group.assign(Resp=group['Resp'].bfill().ffill())

    # 標準化連續欄位：(原始值 - 平均值) / 標準差
    group = group.assign(HR=(group['HR']-df_mean_std['HR']['mean'])/(df_mean_std['HR']['std']))
    group = group.assign(MAP=(group['MAP']-df_mean_std['MAP']['mean'])/(df_mean_std['MAP']['std']))
    group = group.assign(O2Sat=(group['O2Sat']-df_mean_std['O2Sat']['mean'])/(df_mean_std['O2Sat']['std']))
    group = group.assign(SBP=(group['SBP']-df_mean_std['SBP']['mean'])/(df_mean_std['SBP']['std']))
    group = group.assign(Resp=(group['Resp']-df_mean_std['Resp']['mean'])/(df_mean_std['Resp']['std']))

    # 初始化滑動視窗參數
    windowed_data = []
    N = len(group)      # 這個病人總住院時數
    win_len = 10        # 視窗長度：10 小時
    pred_len = 1        # 預測長度：1 小時
    i = 0               # 視窗起始位置

    # 滑動視窗：每次往前移一小時
    while(i+win_len+pred_len <= N):
        # 切割當前視窗輸入（10小時）與預測目標（第11小時）
        tmp_data = group.iloc[i:i+win_len]
        tmp_label = group.iloc[i+win_len:i+win_len+pred_len]
        tmp_label = int(any(tmp_label['SepsisLabel']))
        tmp_patient = patient

        # 視窗往前滑動一小時
        i = i+1

        # 取出連續欄位的時序資料，轉成 numpy array
        X_cont = tmp_data[cols_cont]
        X_cont = X_cont.values

        # 若連續欄位仍有 NaN（補值失敗），跳過此視窗
        if np.isnan(X_cont).any(): continue

        # 處理缺失值多的欄位：取中位數並標準化
        X_binned_dict = {}
        for col_to_bin in cols_to_bin:
            tmp_val = tmp_data[col_to_bin].median()
            # Gender 和 Unit1 是類別變數，不需要標準化
            if col_to_bin not in ['Gender', 'Unit1']:
                tmp_val = (tmp_val-df_mean_std[col_to_bin]['mean'])/df_mean_std[col_to_bin]['std']
            X_binned_dict[col_to_bin] = tmp_val

        # 將所有特徵、標籤、病人 ID 打包成字典
        tmp_dict = X_binned_dict
        tmp_dict['X_cont'] = X_cont      # 連續時序特徵
        tmp_dict['label'] = tmp_label    # 預測標籤
        tmp_dict['patient'] = tmp_patient
        windowed_data.append(tmp_dict)

    # 將這個病人的所有視窗資料轉成 DataFrame 並加入清單
    windowed_data_df = pd.DataFrame(windowed_data)
    windowed_df_list.append(windowed_data_df)

    # 每累積 500 個病人，分割訓練/測試集並存檔
    if (int(patient[-5:]) % 500) == 0:
        print('patient %i' % int(patient[-5:]))
        windowed_df = pd.concat(windowed_df_list).reset_index(drop=True)
        train = windowed_df[windowed_df['patient'].isin(patients_training_data)].drop('patient', axis=1)
        test = windowed_df[~windowed_df['patient'].isin(patients_training_data)].drop('patient', axis=1)

        train.to_pickle('feats/train_%i.pkl' % save_count)
        test.to_pickle('feats/test_%i.pkl' % save_count)

        windowed_df_list = []
        save_count = save_count+1

# 儲存最後一批不足 500 個病人的剩餘資料
if len(windowed_df_list) > 0:
    windowed_df = pd.concat(windowed_df_list).reset_index(drop=True)
    train = windowed_df[windowed_df['patient'].isin(patients_training_data)].drop('patient', axis=1)
    test = windowed_df[~windowed_df['patient'].isin(patients_training_data)].drop('patient', axis=1)
    train.to_pickle('feats/train_%i.pkl' % save_count)
    test.to_pickle('feats/test_%i.pkl' % save_count)

patient 500
patient 1000
patient 1500
patient 2000
patient 2500
patient 3000
patient 3500
patient 4000
patient 4500
patient 5000
patient 5500
patient 6000
patient 6500
patient 7000
patient 7500
patient 8000
patient 8500
patient 9000
patient 9500
patient 10000
patient 10500
patient 11000
patient 11500
patient 12000
patient 12500
patient 13000
patient 13500
patient 14000
patient 14500
patient 15000
patient 15500
patient 16000
patient 16500
patient 17000
patient 17500
patient 18000
patient 18500
patient 19000
patient 19500
patient 20000
patient 20500
patient 500
patient 1000
patient 1500
patient 2000
patient 2500
patient 3000
patient 3500
patient 4000
patient 4500
patient 5000
patient 5500
patient 6000
patient 6500
patient 7000
patient 7500
patient 8000
patient 8500
patient 9000
patient 9500
patient 10000
patient 10500
patient 11000
patient 11500
patient 12000
patient 12500
patient 13000
patient 13500
patient 14000
patient 14500
patient 15000
patient 15500
patient 16000
patient 16500
pati